# publish_risk_current_build
`gold_vuln_daily`에서 **각 CVE의 가장 최근 날짜 한 줄만** 뽑아 `publish_risk_current`로. `gold_circuit_breaker_check` 통과(DAILY) 또는 `gold_vuln_daily_build`(EVENT) 완료 후 트리거.

**Grain 순수성**: CVSS·심각도·패치 버전은 CVE 하나가 여러 GHSA/패키지에 걸치면 대표값을 고를 수 없으므로 **절대 포함하지 않는다** — CVE 단위로 이미 명확한 값(EPSS/KEV/`priority_score`/systemic 여부)만 담는다(03_Gold_Schema.md §3.8, 04_Notebook_설계.md §5.7).

**안 바뀐 CVE는 다시 안 씀**: `record_hash`가 같으면 MERGE의 `whenMatchedUpdate` 조건에서 걸러져 실제 쓰기가 발생하지 않는다.

In [ ]:
%run 00_gold_common

In [ ]:
from pyspark.sql import Window

source_df = spark.table(TBL["gold_vuln_daily"])
if run_mode == "EVENT":
    if not target_cve_ids:
        raise ValueError("EVENT 모드는 target_cve_ids가 필요합니다.")
    source_df = source_df.filter(F.col("cve_id").isin(target_cve_ids))

w = Window.partitionBy("cve_id").orderBy(F.col("date_key").desc())
latest = (
    source_df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)


In [ ]:
priority = (
    spark.table(TBL["ext_priority_score"]).select(
        "cve_id", "priority_score", "priority_score_updated_at"
    )
    if table_exists(TBL["ext_priority_score"]) else None
)

current_df = latest.select(
    "cve_id", "epss_score", "epss_percentile",
    "kev_listed", "kev_date_added", "kev_required_action",
    "is_systemic_day",
)
if priority is not None:
    current_df = current_df.join(priority, on="cve_id", how="left")
else:
    current_df = (
        current_df
        .withColumn("priority_score", F.lit(None).cast(DoubleType()))
        .withColumn("priority_score_updated_at", F.lit(None).cast(StringType()))
    )

CONTENT_COLS = ["epss_score", "epss_percentile", "kev_listed", "kev_date_added",
                 "kev_required_action", "is_systemic_day", "priority_score"]
current_df = (
    current_df
    .withColumn("record_hash", content_hash([F.col(c) for c in CONTENT_COLS]))
    .withColumn("observed_at", F.lit(INGESTED_AT))
)
candidate_count = current_df.count()


In [ ]:
if table_exists(TBL["publish_risk_current"]):
    target = DeltaTable.forName(spark, TBL["publish_risk_current"])
    (
        target.alias("t")
        .merge(current_df.alias("s"), "t.cve_id = s.cve_id")
        .whenMatchedUpdate(condition="t.record_hash <> s.record_hash", set={
            "epss_score": "s.epss_score",
            "epss_percentile": "s.epss_percentile",
            "kev_listed": "s.kev_listed",
            "kev_date_added": "s.kev_date_added",
            "kev_required_action": "s.kev_required_action",
            "is_systemic_day": "s.is_systemic_day",
            "priority_score": "s.priority_score",
            "priority_score_updated_at": "s.priority_score_updated_at",
            "record_hash": "s.record_hash",
            "observed_at": "s.observed_at",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    write_overwrite(current_df, TBL["publish_risk_current"])

log_batch_run("publish_risk_current_build", "SUCCESS", record_count=candidate_count)
print(f"[publish_risk_current_build] candidates={candidate_count} batch_id={batch_id}")
